# 06 · Architecture benchmark (lecturer track, performance only)

**Question.** Is the Transformer earning its place? GRU-D, an LSTM and a minimal RWKV are trained under the same protocol and compared with the models of Phase 2: our softmax Transformer (`softmax_s42`), the KA-Transformer (`ka_s42`) and LightGBM (`gbdt_iculos`).

**This notebook makes no interpretability claim** and none of its models enters notebooks 03–05. It is one of the optional performance analyses of the pre-registration (section 12): FIT and VAL only, no reserved fold.

**Protocol (identical to 01 / 02).** Frozen subject split; FIT-only preprocessing; each stay's most recent 168 h (`runs.predict_store`); same labels and horizons (t+1, t+3, t+6); focal loss; warmup + cosine schedule; early stopping (patience 8, at most 24 epochs); checkpoint soup. Every baseline is given what the Transformer is given: forward-filled values, the observed mask, the time since the last measurement, the static variables and the ICU hour.

**Tuning, stated plainly.** Each baseline gets the same grid of three learning rates, tried at the primary seed and chosen on VAL; every other setting is the Transformer's, and model sizes are fixed in `sepsis_core/baselines.py`. **Our Transformer had 30 Optuna trials on the same VAL fold** (v3), the baselines have 3. Any tuning advantage is therefore the Transformer's, and a baseline that matches it does so with one tenth of the search.

Seeds: 3 per baseline (42, 1042, 2042). The VAL score of the chosen learning rate at seed 42 is a maximum over three runs and is slightly optimistic; the other two seeds are not. Aggregates only.

In [ ]:
# ============================ CELL 1 · BOOTSTRAP: clone the repo, import sepsis_core from it ============================
REPO_URL = "https://github.com/Myakey/sepsis-transformer-analysis.git"
REPO_REF = "main"                                  # branch, tag or commit hash. PIN A COMMIT HASH for any run you will report.
REPO_DIR = "/content/sepsis-transformer-analysis"

import os, sys, subprocess
sys.dont_write_bytecode = True
if "google.colab" in sys.modules and not os.path.exists("/content/drive/MyDrive"):      # Drive holds the data and the checkpoints
    from google.colab import drive
    drive.mount("/content/drive")

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f"git {' '.join(args[:2])} failed: {r.stderr.strip()[-300:]}")
    return r.stdout.strip()

def _auth_url(url):
    tok = None                                     # private repo: Colab secret GITHUB_TOKEN (Notebook access on)
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        tok = os.environ.get("GITHUB_TOKEN")
    return url.replace("https://", f"https://{tok}@") if tok else url

if not os.path.isdir(os.path.join(REPO_DIR, ".git")): _git("clone", "--quiet", _auth_url(REPO_URL), REPO_DIR)
else: _git("fetch", "--quiet", "--all", cwd=REPO_DIR)
_git("checkout", "--quiet", REPO_REF, cwd=REPO_DIR)
if REPO_REF in _git("branch", "--format=%(refname:short)", cwd=REPO_DIR).split(): _git("pull", "--quiet", "--ff-only", cwd=REPO_DIR)
_git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)        # never leave a token in the clone's config
CORE_COMMIT = _git("rev-parse", "HEAD", cwd=REPO_DIR)
os.environ["SEPSIS_PROJECT_ROOT"] = REPO_DIR
os.environ["SEPSIS_CORE_COMMIT"] = CORE_COMMIT
for m in [k for k in sys.modules if k == "sepsis_core" or k.startswith("sepsis_core.")]: del sys.modules[m]
if os.path.join(REPO_DIR, "Notebooks", "v4") not in sys.path: sys.path.insert(0, os.path.join(REPO_DIR, "Notebooks", "v4"))

import importlib.util
_need = [p for p in ("mlflow", "lightgbm") if importlib.util.find_spec(p) is None]
if _need: subprocess.call([sys.executable, "-m", "pip", "install", "-q", *_need])

import json, time, gc
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import sepsis_core
from sepsis_core import config as C, data, dataset, models, engine, metrics, calibration, controls, runs, plots, tracking, attention, physionet, correlational, causal, explain, gbdt, baselines
print("sepsis_core", sepsis_core.__version__, "@ commit", CORE_COMMIT[:12], f"({REPO_REF})", "| torch", torch.__version__)

In [ ]:
# ============================ CELL 2 · SWITCHES ============================
SMOKE_TEST = False
STAGES = dict(grid=True, seeds=True, report=True, by_hour=True, trackD_rows=True)
FAMS = list(baselines.FAMILIES)                      # ["grud", "lstm", "rwkv"]; shorten the list to run one family at a time
LR_GRID = (1e-4, 3e-4, 1e-3)                         # the same three values for every baseline; chosen on VAL at the primary seed

runs.setup(smoke=SMOKE_TEST); baselines.install()
SEEDS = list(C.CFG.SEEDS)[: (2 if SMOKE_TEST else 3)]; PRIMARY = SEEDS[0]; HP = dict(C.V3_BEST_HP); REP_PATH = runs.report_path
if SMOKE_TEST: LR_GRID = LR_GRID[:2]
REFS = {"our Transformer (softmax)": f"softmax_s{PRIMARY}", "KA-Transformer": f"ka_s{PRIMARY}"}            # trained in 02 and 01
TRACKD = {"VarTok (05)": f"vartok_s{PRIMARY}", "kernel attention (02/05)": f"kernel_s{PRIMARY}", "sigmoid attention (02)": f"sigmoid_s{PRIMARY}"}

def have(name, which="best.pt"): return os.path.exists(os.path.join(runs.model_dir(name), which))
def bname(fam, lr, seed): return f"b6_{fam}_lr{lr:g}_s{seed}"
def done_rec(name):
    p = os.path.join(runs.model_dir(name), "done.json"); return json.load(open(p)) if os.path.exists(p) else None
if not have(REFS["our Transformer (softmax)"]): raise FileNotFoundError(f"{REFS['our Transformer (softmax)']} is not in {runs.run_root()}/models. Run notebook 02 first" + (" with SMOKE_TEST = True." if SMOKE_TEST else "."))
print("seeds:", SEEDS, "| learning-rate grid:", LR_GRID, "| families:", {f: baselines.LABELS[f] for f in FAMS})
print("training settings shared with the Transformer:", json.dumps({k: HP[k] for k in ("dropout", "weight_decay", "batch_size", "focal_gamma", "focal_alpha")}),
      "| the Transformer's own learning rate (Optuna, 30 trials):", HP["lr"])

In [ ]:
# ============================ CELL 3 · DATA (FIT/VAL, frozen split, FIT-only preprocessing) AND THE REFERENCE MODELS OF PHASE 2 ============================
B = runs.prepare_mimic(); st = B["store"]
ROWS, SCORES = {}, {}                                # label -> VAL rows / logits, every model on the same rows
r0, l0 = runs.val_scores(REFS["our Transformer (softmax)"]); ROWS0 = r0
def add(label, rows, logits):
    if not np.array_equal(rows, ROWS0): print(f"   [{label}] was scored on different rows than the Transformer -> left out (re-run the notebook that trained it)"); return False
    SCORES[label] = logits; return True
for label, nm in REFS.items():
    if have(nm): add(label, *runs.val_scores(nm))
    else: print(f"   [{label}] {nm} is not in this run folder -> left out (trained by notebook 01)")
G = None
if os.path.exists(os.path.join(runs.model_dir("gbdt_iculos"), "gbdt.json")):
    G = gbdt.fit_or_load(runs.model_dir("gbdt_iculos"), st, B["pre"], B["fit_idx"], B["val_idx"], with_iculos=True); add("LightGBM (+ICULOS)", G["rows"], gbdt.to_logit(G["scores"]))
else: print("   [LightGBM] gbdt_iculos is not in this run folder -> left out (trained by notebook 02)")
print("reference models found:", list(SCORES))

## Learning-rate grid (primary seed), then two more seeds at the chosen value

A run that diverges is recorded as failed for that learning rate; that is a result, not a crash. The choice is saved, so re-running the notebook never changes it.

In [ ]:
# ============================ CELL 4 · GRID: three learning rates per baseline at the primary seed, chosen on VAL AUROC (mean of horizons, soup) ============================
REC, GRID = {}, {}
CHOICE_PATH = REP_PATH("06_lr_choice.json"); CHOICE = json.load(open(CHOICE_PATH)) if os.path.exists(CHOICE_PATH) else {}
def train(fam, lr, seed, role):
    nm = bname(fam, lr, seed)
    try: REC[nm] = runs.train_model(B, nm, baselines.hp_for(fam, HP, lr), seed=seed, role=role); return REC[nm]
    except RuntimeError as e:
        if "diverged" not in str(e) and "no epoch finished" not in str(e): raise
        print(f"[{nm}] did not train: {str(e)[:160]}"); tracking.end_active_runs(); return None
if STAGES["grid"]:
    for fam in FAMS:
        GRID[fam] = {}
        for lr in LR_GRID:
            r = train(fam, lr, PRIMARY, "benchmark_lr_grid"); GRID[fam][f"{lr:g}"] = None if r is None else dict(auroc=r["metrics"]["auroc_mean"], auprc=r["metrics"]["auprc_mean"], epochs=r["epochs_run"])
        ok = {k: v for k, v in GRID[fam].items() if v is not None}
        if not ok: print(f"{fam}: no learning rate of the grid trained"); continue
        if fam not in CHOICE: CHOICE[fam] = max(sorted(ok, key=float), key=lambda k: ok[k]["auroc"])       # ties -> the smaller learning rate
        print(f"{baselines.LABELS[fam]:<40s} " + " | ".join(f"lr {k}: " + ("failed" if v is None else f"AUROC {v['auroc']:.4f} AUPRC {v['auprc']:.4f} ({v['epochs']} ep)") for k, v in GRID[fam].items()) + f"  -> chosen lr {CHOICE[fam]}")
    json.dump(CHOICE, open(CHOICE_PATH, "w"), indent=1)

In [ ]:
# ============================ CELL 5 · SEEDS: the chosen learning rate at the remaining seeds ============================
if STAGES["seeds"]:
    for fam in FAMS:
        if fam not in CHOICE: continue
        for seed in SEEDS: train(fam, float(CHOICE[fam]), seed, "benchmark")
BEST = {fam: [bname(fam, float(CHOICE[fam]), s) for s in SEEDS if have(bname(fam, float(CHOICE[fam]), s))] for fam in FAMS if fam in CHOICE}
for fam, names in BEST.items():
    recs = [done_rec(n) for n in names]; ss = runs.seed_summary(recs)
    print(f"{baselines.LABELS[fam]:<40s} lr {CHOICE[fam]} | VAL AUROC over {ss['n']} seeds: {ss['mean']:.4f} +/- {ss['sd']:.4f} [{ss['min']:.4f}, {ss['max']:.4f}] | parameters {recs[0]['n_params']:,} | "
          f"epoch {np.mean([r['epoch_sec'] for r in recs]):.0f}s | epochs run {[r['epochs_run'] for r in recs]}")
    if names and names[0].endswith(f"_s{PRIMARY}"): add(baselines.LABELS[fam], *runs.val_scores(names[0]))

## The comparison (VAL, primary seed, every model on the same rows)

AUROC and AUPRC per horizon with stay-level bootstrap CIs; then each model minus our Transformer with a stay-level paired bootstrap. A difference smaller than the seed SD of the Transformer (0.003 in notebook 02) is "not resolved".

In [ ]:
# ============================ CELL 6 · REPORT: AUROC / AUPRC per horizon with CIs, paired difference vs our Transformer, size and speed ============================
REPORT, PAIRED, INFO = {}, {}, {}
if STAGES["report"]:
    for label, lg in SCORES.items():
        REPORT[label] = runs.report(st, ROWS0, lg, B["onset_rows"], hours=True); print(runs.line(label, REPORT[label]))
    print("\nper horizon [point, 95% CI by stay]:")
    for label, rep in REPORT.items():
        print(f"   {label:<40s} AUROC " + " | ".join(f"t+{h} {rep['auroc_ci'][f't{h}'][0]:.4f} [{rep['auroc_ci'][f't{h}'][1]:.4f}, {rep['auroc_ci'][f't{h}'][2]:.4f}]" for h in C.HORIZONS))
        print(f"   {'':<40s} AUPRC " + " | ".join(f"t+{h} {rep['auprc_ci'][f't{h}'][0]:.4f} [{rep['auprc_ci'][f't{h}'][1]:.4f}, {rep['auprc_ci'][f't{h}'][2]:.4f}]" for h in C.HORIZONS))
    ref = "our Transformer (softmax)"
    print(f"\nmodel minus {ref}, AUROC, stay-level paired bootstrap on the same VAL rows (positive = the model is ahead):")
    for label, lg in SCORES.items():
        if label == ref: continue
        PAIRED[label] = runs.paired(st, ROWS0, SCORES[ref], lg); p = PAIRED[label]
        print(f"   {label:<40s} mean of horizons {p['d']:+.4f} [{p['lo']:+.4f}, {p['hi']:+.4f}] | " + " | ".join(f"t+{h} {p['per_horizon'][f't{h}'][0]:+.4f} [{p['per_horizon'][f't{h}'][1]:+.4f}, {p['per_horizon'][f't{h}'][2]:+.4f}]" for h in C.HORIZONS)
              + f" | {'baseline ahead' if p['lo'] > 0 else 'Transformer ahead' if p['hi'] < 0 else 'no clear difference'}")
    print("\nsame-seed differences for the other seeds (baseline minus softmax of that seed):")
    for fam, names in BEST.items():
        ds = []
        for n in names:
            seed = int(n.rsplit("_s", 1)[1]); sm = f"softmax_s{seed}"
            if not have(sm): continue
            ra, a = runs.val_scores(sm); rb, b = runs.val_scores(n)
            if np.array_equal(ra, rb): ds.append((seed, runs.paired(st, ra, a, b)))
        PAIRED[f"{fam}__by_seed"] = {s: dict(d=p["d"], lo=p["lo"], hi=p["hi"]) for s, p in ds}
        if ds: print(f"   {baselines.LABELS[fam]:<40s} " + " | ".join(f"seed {s}: {p['d']:+.4f} [{p['lo']:+.4f}, {p['hi']:+.4f}]" for s, p in ds) + f" | mean {np.mean([p['d'] for _, p in ds]):+.4f}")
    for label, nm in list(REFS.items()) + [(baselines.LABELS[f], n[0]) for f, n in BEST.items() if n]:
        r = done_rec(nm)
        if r: INFO[label] = dict(model=nm, n_params=int(r["n_params"]), epoch_sec=float(r["epoch_sec"]) if r.get("epoch_sec") else None, epochs_run=int(r["epochs_run"]), lr=float(r["hp"]["lr"]))
    if G is not None: INFO["LightGBM (+ICULOS)"] = dict(model="gbdt_iculos", n_params=None, epoch_sec=None, epochs_run=None, lr=None, trees=G.get("best_iter"))
    print("\nsize and speed (epoch time depends on the GPU of the session that trained the model; compare only runs from the same kind of session):")
    for label, i in INFO.items():
        print(f"   {label:<40s} {i['model']:<24s} parameters {('%s' % format(i['n_params'], ',')) if i['n_params'] else 'n/a':>12s} | epoch {('%.0fs' % i['epoch_sec']) if i['epoch_sec'] else 'n/a':>6s} | epochs {i['epochs_run']} | lr {i['lr']}" + (f" | trees {i['trees']}" if i.get("trees") else ""))

In [ ]:
# ============================ CELL 7 · BY ICU HOUR: how early can each model tell? (AUROC at t+6, all stays and with early-onset stays removed) ============================
BYHOUR = {}
if STAGES["by_hour"] and REPORT:
    for view in ("all", "no_early_onset"):
        buckets = [r["bucket"] for r in next(iter(REPORT.values()))["by_icu_hour"] if r["view"] == view]
        print(f"\nAUROC at t+6 by ICU-hour bucket, view '{view}'" + (" (stays with onset before hour 12 removed: in those, the first hours are recognition, not prediction)" if view != "all" else "") + ":")
        print(f"   {'model':<40s} " + " ".join(f"{b:>9s}" for b in buckets))
        for label, rep in REPORT.items():
            vals = {r["bucket"]: r["auroc_t6"] for r in rep["by_icu_hour"] if r["view"] == view}; BYHOUR.setdefault(view, {})[label] = vals
            print(f"   {label:<40s} " + " ".join(f"{vals[b]:>9.4f}" for b in buckets))
    rows_n = {r["bucket"]: r["n_rows"] for r in next(iter(REPORT.values()))["by_icu_hour"] if r["view"] == "all"}
    print("   rows per bucket (all stays):", rows_n)
    plots.style(); plots.by_icu_hour({k: v["by_icu_hour"] for k, v in REPORT.items()}, horizon=6, metric="auroc", path=runs.figure_path("06_auroc_by_hour_t6.png")); plt.show()

In [ ]:
# ============================ CELL 8 · FOR THE REPORT ONLY: the Track D models, if notebook 05 has trained them (not part of the benchmark) ============================
TRACKD_ROWS = {}
if STAGES["trackD_rows"]:
    ref = "our Transformer (softmax)"
    for label, nm in TRACKD.items():
        if not have(nm) or not os.path.exists(os.path.join(runs.model_dir(nm), "val_scores.npz")): print(f"   [{label}] {nm} not found -> row left empty (run notebook 05 first if you want it in the table)"); continue
        rows, lg = runs.val_scores(nm)
        if not np.array_equal(rows, ROWS0): print(f"   [{label}] scored on different rows -> left out"); continue
        rep = runs.report(st, rows, lg, B["onset_rows"], hours=False); p = runs.paired(st, rows, SCORES[ref], lg); r = done_rec(nm) or {}
        TRACKD_ROWS[label] = dict(model=nm, report=rep, paired=p, n_params=r.get("n_params"), epoch_sec=r.get("epoch_sec"))
        print(runs.line(label, rep) + f" | minus Transformer {p['d']:+.4f} [{p['lo']:+.4f}, {p['hi']:+.4f}]")

In [ ]:
# ============================ CELL 9 · SAVE, AND THE TABLE TO PASTE INTO THE REPORT (this is what to send back) ============================
def cell(ci): return f"{ci[0]:.3f} [{ci[1]:.3f}, {ci[2]:.3f}]"
L = ["| Model | Parameters | AUROC t+1 | AUROC t+3 | AUROC t+6 | AUPRC t+6 | AUROC minus our Transformer (mean of horizons) | Seeds: mean ± SD | Epoch |", "|---|---|---|---|---|---|---|---|---|"]
for label, rep in REPORT.items():
    i = INFO.get(label, {}); p = PAIRED.get(label); fam = next((f for f in BEST if baselines.LABELS[f] == label), None)
    ss = runs.seed_summary([done_rec(n) for n in BEST[fam]]) if fam else None
    L.append(f"| {label} | {format(i['n_params'], ',') if i.get('n_params') else 'n/a'} | {cell(rep['auroc_ci']['t1'])} | {cell(rep['auroc_ci']['t3'])} | {cell(rep['auroc_ci']['t6'])} | {cell(rep['auprc_ci']['t6'])} | "
             f"{'reference' if p is None else '%+.4f [%+.4f, %+.4f]' % (p['d'], p['lo'], p['hi'])} | {('%.4f ± %.4f (n=%d)' % (ss['mean'], ss['sd'], ss['n'])) if ss else 'see notebook 02'} | {('%.0f s' % i['epoch_sec']) if i.get('epoch_sec') else 'n/a'} |")
for label, t in TRACKD_ROWS.items():
    rep, p = t["report"], t["paired"]
    L.append(f"| {label} (Track D, not a benchmark entry) | {format(t['n_params'], ',') if t.get('n_params') else 'n/a'} | {cell(rep['auroc_ci']['t1'])} | {cell(rep['auroc_ci']['t3'])} | {cell(rep['auroc_ci']['t6'])} | {cell(rep['auprc_ci']['t6'])} | "
             f"{p['d']:+.4f} [{p['lo']:+.4f}, {p['hi']:+.4f}] | see notebook 05 | {('%.0f s' % t['epoch_sec']) if t.get('epoch_sec') else 'n/a'} |")
note = (f"VAL fold, most recent 168 h per stay, primary seed {PRIMARY}; CIs by stay-level bootstrap; differences by stay-level paired bootstrap. Baselines: learning rate chosen on VAL from {list(LR_GRID)} "
        f"(chosen: {CHOICE}); our Transformer: 30 Optuna trials on the same fold. Performance only: no interpretability claim.")
open(REP_PATH("06_table.md"), "w").write("\n".join(L) + "\n\n" + note + "\n"); print("\n".join(L)); print("\n" + note)
runs.save_json(dict(core_commit=CORE_COMMIT, run=C.CFG.RUN_NAME, smoke=SMOKE_TEST, seeds=SEEDS, lr_grid=list(LR_GRID), grid=GRID, lr_choice=CHOICE, models=BEST, report=REPORT, paired=PAIRED, info=INFO,
                    by_icu_hour_auroc_t6=BYHOUR, trackD_rows=TRACKD_ROWS, transformer_tuning="30 Optuna trials on VAL (v3)", baseline_tuning=f"{len(LR_GRID)} learning rates on VAL"), "06_summary.json")

## What to send back

The printed output of Cells 4–9, or `reports/06_summary.json` and `reports/06_table.md`. All of it is aggregate.

Reading it (the same rule as notebook 02): a difference whose CI includes 0, or that is smaller than the seed SD, is "no clear difference". Nothing here was executed before hand-over: expect to fix small things on the first smoke run and send me the traceback.